# Đánh giá agent: ReAct tự viết, native tool calling, JSON schema

Một agent "chạy được" trên ba câu thử chưa nói lên gì. Bài này đo agent như đo một model: **bộ task ghim** có đáp án tính
từ dữ liệu (24 câu hỏi về một hiệu sách tự tạo, `bookstore.py`), chấm tự động, khoảng tin cậy cho tỉ lệ thành công, so
theo cặp giữa hai cách, và **phân loại lỗi** tự động để biết agent hỏng ở đâu. Ba cách gọi tool trên cùng `qwen2.5:3b`:

- **ReAct văn bản**: vòng `run_react` của bạn (bài Basic 02), model viết `Thought / Action / Action Input`.
- **Native**: trường `tools` của Ollama, chat template của Qwen2.5 lo định dạng (module 11 mục 6).
- **ReAct JSON**: cùng vòng suy nghĩ nhưng mỗi lượt ép output bằng JSON schema (`format` của Ollama).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b`. Các cell `explore` gọi model khoảng 320 lượt, khoảng 10 phút trên PC của track.
- Chấm: `uv run pytest tracks/ai-engineer/modules/16-ai-agents/exercises/intermediate/02-agent-eval -q`

In [ ]:
# explore: đưa gốc repo vào sys.path để import llmkit (test chạy từ gốc repo nên không cần cell này)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import math
import re
import time
import unicodedata
from dataclasses import asdict

import pandas as pd

import bookstore

## 1. Dán agent của bạn

Notebook tự chứa: dán lời giải bài Basic 01 (`ToolRegistry` và các hàm đi kèm) và bài Basic 02 (`parse_react`,
`run_react` và các dataclass) vào cell dưới. Test của bài này không chấm cell đó, nhưng mọi cell `explore` dùng nó.

In [ ]:
# Lời giải bài Basic 01 (tools.py)
import inspect
import json
import re
import types
import typing
from dataclasses import dataclass
from typing import Callable, Literal

import jsonschema


@dataclass
class Observation:
    ok: bool
    content: str


_PRIMITIVES = {int: "integer", float: "number", str: "string", bool: "boolean"}


def schema_for(annotation, where: str = "") -> dict:
    if annotation in _PRIMITIVES:
        return {"type": _PRIMITIVES[annotation]}
    origin = typing.get_origin(annotation)
    args = typing.get_args(annotation)
    if origin is Literal:
        kinds = {type(v) for v in args}
        if kinds == {str}:
            return {"type": "string", "enum": list(args)}
        if kinds == {int}:
            return {"type": "integer", "enum": list(args)}
        raise TypeError(f"{where}: Literal chỉ nhận toàn str hoặc toàn int")
    if origin is list:
        if len(args) != 1:
            raise TypeError(f"{where}: list cần đúng một kiểu phần tử, ví dụ list[str]")
        return {"type": "array", "items": schema_for(args[0], where)}
    if origin in (typing.Union, types.UnionType):
        inner = [a for a in args if a is not type(None)]
        if len(args) == 2 and len(inner) == 1:
            schema = dict(schema_for(inner[0], where))
            schema["type"] = [schema["type"], "null"]
            if "enum" in schema:
                schema["enum"] = [*schema["enum"], None]
            return schema
        raise TypeError(f"{where}: Union chỉ hỗ trợ dạng `T | None`")
    raise TypeError(f"{where}: kiểu {annotation!r} chưa hỗ trợ")


_ARG = re.compile(r"^\s*(\w+)\s*(?:\([^)]*\))?\s*:\s*(.+)$")


def parse_docstring(doc: str | None) -> tuple[str, dict[str, str]]:
    doc = inspect.cleandoc(doc or "")
    head, _, rest = doc.partition("Args:")
    description = " ".join(line.strip() for line in head.splitlines() if line.strip())
    params: dict[str, str] = {}
    current = None
    for line in rest.splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and line.rstrip().endswith(":"):
            break
        match = _ARG.match(line)
        if match:
            current = match.group(1)
            params[current] = match.group(2).strip()
        elif current:
            params[current] += " " + line.strip()
    return description, params


def tool_spec(fn: Callable) -> dict:
    name = fn.__name__
    description, docs = parse_docstring(fn.__doc__)
    hints = typing.get_type_hints(fn)
    properties: dict[str, dict] = {}
    required: list[str] = []
    for pname, param in inspect.signature(fn).parameters.items():
        if param.kind in (param.VAR_POSITIONAL, param.VAR_KEYWORD):
            raise TypeError(f"{name}: không hỗ trợ *args/**kwargs ({pname})")
        if pname not in hints:
            raise TypeError(f"{name}: tham số {pname} thiếu type hint")
        schema = schema_for(hints[pname], f"{name}.{pname}")
        if pname in docs:
            schema["description"] = docs[pname]
        if param.default is inspect.Parameter.empty:
            required.append(pname)
        elif param.default is not None:
            schema["default"] = param.default
        properties[pname] = schema
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required,
                       "additionalProperties": False}}}


class ToolRegistry:
    def __init__(self):
        self._fns: dict[str, Callable] = {}
        self._specs: dict[str, dict] = {}

    def register(self, fn: Callable) -> Callable:
        spec = tool_spec(fn)
        name = spec["function"]["name"]
        if name in self._fns:
            raise ValueError(f"tool {name!r} đã được đăng ký")
        self._fns[name], self._specs[name] = fn, spec
        return fn

    def specs(self) -> list[dict]:
        return list(self._specs.values())

    def call(self, name: str, arguments) -> Observation:
        if name not in self._fns:
            return Observation(False, f"ERROR: không có tool {name!r}. Tool hợp lệ: {', '.join(self._fns)}.")
        args = arguments
        if isinstance(arguments, str):
            try:
                args = json.loads(arguments)
            except ValueError:
                args = None
        if not isinstance(args, dict):
            return Observation(False, f"ERROR: tham số của {name} phải là một JSON object, nhận {arguments!r}.")
        validator = jsonschema.Draft202012Validator(self._specs[name]["function"]["parameters"])
        errors = sorted(validator.iter_errors(args), key=lambda e: [str(p) for p in e.path])
        if errors:
            detail = "; ".join((".".join(map(str, e.path)) + ": " if e.path else "") + e.message for e in errors)
            return Observation(False, f"ERROR: tham số của {name} sai: {detail}")
        try:
            result = self._fns[name](**args)
        except Exception as exc:
            return Observation(False, f"ERROR: {name} lỗi khi chạy: {type(exc).__name__}: {exc}")
        if isinstance(result, str):
            return Observation(True, result)
        return Observation(True, json.dumps(result, ensure_ascii=False, default=str))


# Lời giải bài Basic 02 (react.py)
import time
from dataclasses import field

import llmkit


@dataclass
class Action:
    tool: str
    args: dict
    thought: str = ""


@dataclass
class Final:
    answer: str
    thought: str = ""


class ReActParseError(ValueError):
    pass


@dataclass
class Step:
    index: int
    kind: str
    reply: str
    tool: str | None = None
    args: dict | None = None
    observation: str | None = None
    ok: bool | None = None
    seconds: float = 0.0


@dataclass
class RunResult:
    answer: str | None
    stopped: str
    steps: list[Step] = field(default_factory=list)
    messages: list[dict] = field(default_factory=list)
    tool_errors: int = 0
    format_errors: int = 0
    error: str | None = None


SYSTEM_TEMPLATE = """Bạn là một agent giải quyết yêu cầu của người dùng bằng cách dùng tool. Các tool có sẵn:

{tools}

Mỗi lượt, trả lời đúng MỘT trong hai dạng sau, không viết gì khác:

Thought: <suy nghĩ ngắn: cần làm gì tiếp>
Action: <tên tool>
Action Input: <tham số dạng JSON object>

hoặc, khi đã đủ thông tin:

Thought: <suy nghĩ ngắn>
Final Answer: <câu trả lời cho người dùng, tiếng Việt>

Sau mỗi Action, hệ thống trả về `Observation: <kết quả>`. Không tự viết Observation. Số liệu phải lấy từ Observation,
không tự bịa."""


def render_tools(specs: list[dict]) -> str:
    lines = []
    for spec in specs:
        f = spec["function"]
        lines.append(f"- {f['name']}: {f['description']}\n  Tham số (JSON Schema): "
                     f"{json.dumps(f['parameters'], ensure_ascii=False)}")
    return "\n".join(lines)


_OBSERVATION = re.compile(r"^\s*Observation\s*:", re.I | re.M)
_ACTION = re.compile(r"^\s*Action\s*:[ \t]*(.*)$", re.I | re.M)
_INPUT = re.compile(r"^\s*Action\s+Input\s*:", re.I | re.M)
_FINAL = re.compile(r"^\s*Final\s+Answer\s*:", re.I | re.M)
_THOUGHT = re.compile(r"^\s*Thought\s*:(.*?)(?=^\s*(?:Action|Final\s+Answer)\b|\Z)", re.I | re.M | re.S)
_FENCE = re.compile(r"^```[a-zA-Z]*\s*|\s*```\s*$")


def clean_reply(text: str) -> str:
    match = _OBSERVATION.search(text)
    return (text[: match.start()] if match else text).strip()


def parse_react(text: str) -> Action | Final:
    text = clean_reply(text)
    thought_match = _THOUGHT.search(text)
    thought = thought_match.group(1).strip() if thought_match else ""
    action = _ACTION.search(text)
    final = _FINAL.search(text)
    if action and (not final or action.start() < final.start()):
        tool = action.group(1).strip().strip("`'\"*").strip()
        if not tool:
            raise ReActParseError("Dòng `Action:` thiếu tên tool.")
        inp = _INPUT.search(text, action.end())
        if not inp:
            raise ReActParseError(f"Thiếu dòng `Action Input:` (JSON object) cho tool {tool}.")
        end = final.start() if final and final.start() > inp.end() else len(text)
        raw = _FENCE.sub("", text[inp.end():end].strip()).strip()
        try:
            args, _ = json.JSONDecoder().raw_decode(raw)
        except ValueError:
            raise ReActParseError(f"`Action Input` của {tool} không phải JSON hợp lệ: {raw[:80]!r}.") from None
        if not isinstance(args, dict):
            raise ReActParseError(f"`Action Input` của {tool} phải là JSON object, nhận {raw[:80]!r}.")
        return Action(tool, args, thought)
    if final:
        answer = text[final.end():].strip()
        if not answer:
            raise ReActParseError("`Final Answer:` rỗng.")
        return Final(answer, thought)
    raise ReActParseError("Không thấy `Action:` hoặc `Final Answer:`.")


def run_react(llm, task: str, registry, *, max_steps: int = 8, max_format_retries: int = 2, max_repeats: int = 2,
              system: str | None = None, history: list[dict] | None = None) -> RunResult:
    prompt = (system or SYSTEM_TEMPLATE).format(tools=render_tools(registry.specs()))
    messages = [{"role": "system", "content": prompt}, *[dict(m) for m in history or []],
                {"role": "user", "content": task}]
    result = RunResult(None, "max_steps", [], messages)
    consecutive_format = 0
    last_key = None
    repeats = 0
    for index in range(max_steps):
        start = time.perf_counter()
        try:
            reply = llm.chat(messages)
        except llmkit.LLMError as exc:
            result.stopped, result.error = "llm_error", str(exc)
            return result
        try:
            parsed = parse_react(reply)
        except ReActParseError as exc:
            consecutive_format += 1
            result.format_errors += 1
            messages.append({"role": "assistant", "content": reply})
            messages.append({"role": "user", "content": f"Lỗi định dạng: {exc} Trả lời lại đúng định dạng."})
            result.steps.append(Step(index, "format_error", reply, observation=str(exc), ok=False,
                                     seconds=time.perf_counter() - start))
            if consecutive_format > max_format_retries:
                result.stopped = "format_error"
                return result
            continue
        consecutive_format = 0
        cleaned = clean_reply(reply)
        if isinstance(parsed, Final):
            messages.append({"role": "assistant", "content": cleaned})
            result.steps.append(Step(index, "final", reply, seconds=time.perf_counter() - start))
            result.answer, result.stopped = parsed.answer, "answer"
            return result
        key = (parsed.tool, json.dumps(parsed.args, sort_keys=True, ensure_ascii=False))
        repeats = repeats + 1 if key == last_key else 0
        last_key = key
        if repeats >= max_repeats:
            result.steps.append(Step(index, "repeat", reply, parsed.tool, parsed.args, ok=False,
                                     seconds=time.perf_counter() - start))
            result.stopped = "repeated_action"
            return result
        obs = registry.call(parsed.tool, parsed.args)
        result.tool_errors += not obs.ok
        messages.append({"role": "assistant", "content": cleaned})
        messages.append({"role": "user", "content": f"Observation: {obs.content}"})
        result.steps.append(Step(index, "action", reply, parsed.tool, parsed.args, obs.content, obs.ok,
                                 time.perf_counter() - start))
    return result

## 2. Chấm câu trả lời

Câu trả lời của agent là chữ tự do: "Tổng là 165.000 đồng", "khoảng 5,31 USD", "2,3 triệu". `numbers_in(text)` trả mọi
con số trong chuỗi theo thứ tự xuất hiện, kiểu `float`, theo quy ước viết số tiếng Việt:

- Nhóm 3 chữ số cách nhau bởi `.`, `,` hoặc dấu cách sau 1 tới 3 chữ số đầu là **phân cách hàng nghìn**:
  `1.234.567`, `165,000`, `1 000 000`.
- Một dấu `.` hoặc `,` không theo dạng trên là **dấu thập phân**: `5,31`, `2.84`, `0.8`.
- Số theo sau bởi ` triệu` nhân $$10^6$$, bởi ` nghìn` hoặc ` ngàn` nhân $$10^3$$ (có hoặc không có dấu cách).
- Không đọc số âm; chữ số dính chữ cái vẫn tính (`B19` cho 19).

`check_answer(answer, expected)`: `answer` rỗng hoặc `None` thì `False`. `expected` là list: phần tử chuỗi phải xuất
hiện trong câu trả lời (không phân biệt hoa thường, sau chuẩn hoá NFC); phần tử số phải khớp **một** số trong
`numbers_in(answer)` với sai lệch tuyệt đối không quá `0.01`. Mọi phần tử phải đạt.

In [ ]:
_NUMBER = re.compile(r"(\d{1,3}(?:[., ]\d{3})+(?!\d)|\d+(?:[.,]\d+)?)(\s*(?:triệu|nghìn|ngàn))?")


def numbers_in(text: str) -> list[float]:
    """Mọi con số trong text theo thứ tự xuất hiện (quy ước ở trên)."""
    out = []
    for raw, unit in _NUMBER.findall(unicodedata.normalize("NFC", text or "")):
        if re.fullmatch(r"\d{1,3}(?:[., ]\d{3})+", raw):
            value = float(re.sub(r"[., ]", "", raw))
        else:
            value = float(raw.replace(",", "."))
        unit = unit.strip()
        if unit == "triệu":
            value *= 1_000_000
        elif unit in ("nghìn", "ngàn"):
            value *= 1_000
        out.append(value)
    return out


def check_answer(answer: str | None, expected: list) -> bool:
    """Mọi phần tử của expected đều có trong answer (chuỗi: chứa, không phân biệt hoa thường; số: sai lệch <= 0.01)."""
    if not answer:
        return False
    text = unicodedata.normalize("NFC", answer).casefold()
    found = numbers_in(answer)
    for item in expected:
        if isinstance(item, str):
            if unicodedata.normalize("NFC", item).casefold() not in text:
                return False
        elif not any(abs(n - item) <= 0.01 for n in found):
            return False
    return True

## 3. Tỉ lệ thành công có khoảng tin cậy, so theo cặp, phân loại lỗi

`wilson_interval(k, n, z=1.96)`: khoảng tin cậy Wilson cho tỉ lệ $$\hat p = k/n$$ (mục 2 lý thuyết), trả `(lo, hi)`;
`n = 0` thì `(0.0, 1.0)`.

$$
\frac{\hat p + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}
$$

`discordant(a, b)`: `a`, `b` là dict `task_id -> bool` của hai cách trên **cùng** bộ task (khoá khác nhau thì
`ValueError`). Trả `(chỉ_a_đúng, chỉ_b_đúng)`, mỗi phần là list id đã sắp xếp. Task cả hai cùng đúng hoặc cùng sai
không nói gì về chênh lệch; chỉ các cặp lệch nhau mới nói (ý của kiểm định McNemar).

`classify_failure(row)`: `row` là dict có `ok`, `stopped`, `answer`, `question`, `steps` (list dict có `kind`,
`observation`, `ok`, như `asdict(Step)`). Trả nhãn đầu tiên khớp theo thứ tự:

1. `"ok"`: `row["ok"]` đúng.
2. `"no_answer"`: `stopped` khác `"answer"` (hết bước, lỗi định dạng, lặp).
3. `"no_tool"`: không có bước nào `kind == "action"`.
4. `"hallucinated_number"`: agent dùng một con số không lấy từ đâu cả. Duyệt các bước theo thứ tự, giữ danh sách số "đã
   thấy" (ban đầu là số trong câu hỏi; sau mỗi bước action thêm số trong observation của nó). Một bước action có số
   trong tham số (`numbers_in(json.dumps(args, ensure_ascii=False))`) chưa thấy trước bước đó, hoặc câu trả lời cuối có
   số chưa thấy sau mọi bước, là nhãn này. "Chưa thấy" nghĩa là sai lệch quá `0.01` với mọi số đã thấy. Số bịa thường
   đi vào **tham số** của `calculator` rồi ra observation như một kết quả hợp lệ; chỉ kiểm câu trả lời là bỏ sót nó.
5. `"tool_error"`: có bước action với `ok` sai.
6. `"wrong_other"`: còn lại.

In [ ]:
def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return 0.0, 1.0
    p = k / n
    denom = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def discordant(a: dict[str, bool], b: dict[str, bool]) -> tuple[list[str], list[str]]:
    if set(a) != set(b):
        raise ValueError("hai cách phải chạy trên cùng bộ task")
    return sorted(t for t in a if a[t] and not b[t]), sorted(t for t in a if b[t] and not a[t])


def classify_failure(row: dict) -> str:
    if row["ok"]:
        return "ok"
    if row["stopped"] != "answer":
        return "no_answer"
    actions = [s for s in row["steps"] if s["kind"] == "action"]
    if not actions:
        return "no_tool"
    seen = numbers_in(row["question"])

    def unseen(numbers):
        return any(all(abs(n - m) > 0.01 for m in seen) for n in numbers)

    for s in actions:
        if unseen(numbers_in(json.dumps(s.get("args") or {}, ensure_ascii=False))):
            return "hallucinated_number"
        seen += numbers_in(s.get("observation") or "")
    if unseen(numbers_in(row["answer"])):
        return "hallucinated_number"
    if any(s["ok"] is False for s in actions):
        return "tool_error"
    return "wrong_other"

## 4. Chạy thật trên 24 task

Ba cách dùng chung `ToolRegistry` với 5 tool của `bookstore.py`, `temperature` 0, `seed` 0, tối đa 8 bước. Client
nhỏ dưới đây gọi thẳng `/api/chat` của Ollama (giống `llmkit` nhưng có thêm `seed`, `tools` và đếm token), kèm vòng
native và vòng ReAct JSON (đây là phần harness, không phải bài tập). Cả ba cách chạy khoảng 8 phút trên PC của
track; cách JSON chậm nhất.

In [ ]:
# explore: client Ollama có seed, vòng native và vòng ReAct JSON
import httpx

import llmkit

OLLAMA = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MODEL = "qwen2.5:3b"


class OllamaChat:
    def __init__(self, model=MODEL, seed=0, num_ctx=None):
        self.model, self.seed, self.num_ctx = model, seed, num_ctx
        self.http = httpx.Client(timeout=600)
        self.prompt_tokens = []

    def raw(self, messages, tools=None, json_schema=None):
        options = {"temperature": 0, "seed": self.seed}
        if self.num_ctx:
            options["num_ctx"] = self.num_ctx
        payload = {"model": self.model, "messages": messages, "stream": False, "options": options}
        if tools:
            payload["tools"] = tools
        if json_schema is not None:
            payload["format"] = json_schema
        r = self.http.post(OLLAMA + "/api/chat", json=payload)
        if r.status_code >= 400:
            raise llmkit.LLMError(f"HTTP {r.status_code}: {r.text[:200]}")
        data = r.json()
        self.prompt_tokens.append(data.get("prompt_eval_count") or 0)
        return data["message"]

    def chat(self, messages, *, model=None, temperature=0.0, json_schema=None):
        return self.raw(messages, json_schema=json_schema)["content"]


NATIVE_SYSTEM = ("Bạn là trợ lý của một hiệu sách. Dùng tool khi cần số liệu; số liệu phải lấy từ kết quả tool. "
                 "Trả lời tiếng Việt, ngắn gọn.")


def run_native(client, task, registry, max_steps=8):
    messages = [{"role": "system", "content": NATIVE_SYSTEM}, {"role": "user", "content": task}]
    result = RunResult(None, "max_steps", [], messages)
    for index in range(max_steps):
        start = time.perf_counter()
        msg = client.raw(messages, tools=registry.specs())
        calls = msg.get("tool_calls") or []
        if not calls:
            result.answer, result.stopped = msg.get("content", ""), "answer"
            result.steps.append(Step(index, "final", msg.get("content", ""), seconds=time.perf_counter() - start))
            return result
        messages.append({"role": "assistant", "content": msg.get("content", ""),
                         "tool_calls": [{"function": c["function"]} for c in calls]})
        for c in calls:
            name, args = c["function"]["name"], c["function"].get("arguments", {})
            obs = registry.call(name, args)
            result.tool_errors += not obs.ok
            messages.append({"role": "tool", "content": obs.content, "tool_name": name})
            result.steps.append(Step(index, "action", json.dumps(c["function"], ensure_ascii=False), name,
                                     args if isinstance(args, dict) else None, obs.content, obs.ok,
                                     time.perf_counter() - start))
    return result


JSON_SYSTEM = SYSTEM_TEMPLATE.split("Mỗi lượt")[0] + (
    'Mỗi lượt, trả lời bằng MỘT JSON object:\n'
    '- cần gọi tool: {{"thought": "...", "action": "<tên tool>", "action_input": {{...}}}}\n'
    '- đã đủ thông tin: {{"thought": "...", "action": "Final Answer", "final_answer": "<câu trả lời tiếng Việt>"}}\n'
    "Sau mỗi action, hệ thống trả về `Observation: <kết quả>`. Số liệu phải lấy từ Observation, không tự bịa.")


def run_react_json(client, task, registry, max_steps=8):
    names = [s["function"]["name"] for s in registry.specs()] + ["Final Answer"]
    schema = {"type": "object", "required": ["thought", "action"], "properties": {
        "thought": {"type": "string"}, "action": {"type": "string", "enum": names},
        "action_input": {"type": "object"}, "final_answer": {"type": "string"}}}
    messages = [{"role": "system", "content": JSON_SYSTEM.format(tools=render_tools(registry.specs()))},
                {"role": "user", "content": task}]
    result = RunResult(None, "max_steps", [], messages)
    for index in range(max_steps):
        start = time.perf_counter()
        reply = client.chat(messages, json_schema=schema)
        messages.append({"role": "assistant", "content": reply})
        try:
            data = json.loads(reply)
            action = data["action"]
        except (ValueError, KeyError, TypeError):
            result.format_errors += 1
            messages.append({"role": "user", "content": "Lỗi định dạng: cần JSON object có thought, action."})
            result.steps.append(Step(index, "format_error", reply, seconds=time.perf_counter() - start))
            continue
        if action == "Final Answer":
            result.answer, result.stopped = data.get("final_answer", ""), "answer"
            result.steps.append(Step(index, "final", reply, seconds=time.perf_counter() - start))
            return result
        obs = registry.call(action, data.get("action_input", {}))
        result.tool_errors += not obs.ok
        messages.append({"role": "user", "content": f"Observation: {obs.content}"})
        result.steps.append(Step(index, "action", reply, action, data.get("action_input"), obs.content, obs.ok,
                                 time.perf_counter() - start))
    return result


registry = ToolRegistry()
for fn in bookstore.TOOLS:
    registry.register(fn)
print([s["function"]["name"] for s in registry.specs()])
print(len(bookstore.TASKS), "task; ví dụ:", bookstore.TASKS[12])

In [ ]:
# explore: chạy 3 cách x 24 task (khoảng 8 phút trên PC của track)
MODES = {
    "react": lambda client, q: run_react(client, q, registry, max_steps=8),
    "native": lambda client, q: run_native(client, q, registry, max_steps=8),
    "react_json": lambda client, q: run_react_json(client, q, registry, max_steps=8),
}
rows = []
for mode, agent in MODES.items():
    for task_id, group, question, expected in bookstore.TASKS:
        client = OllamaChat()
        start = time.perf_counter()
        res = agent(client, question)
        rows.append(dict(mode=mode, id=task_id, group=group, question=question, expected=expected,
                         ok=check_answer(res.answer, expected), stopped=res.stopped, answer=res.answer or "",
                         steps=[asdict(s) for s in res.steps], n_steps=len(res.steps), tool_errors=res.tool_errors,
                         format_errors=res.format_errors, seconds=time.perf_counter() - start,
                         prompt_tokens=max(client.prompt_tokens)))
        r = rows[-1]
        print(f"{mode:10s} {task_id} {'ĐÚNG' if r['ok'] else 'sai '} {r['stopped']:15s} {r['n_steps']} bước "
              f"{r['seconds']:5.1f}s | {r['answer'][:80]!r}")
df = pd.DataFrame(rows)

In [ ]:
# explore: bảng tổng hợp có khoảng tin cậy, so theo cặp
summary = []
for mode, g in df.groupby("mode", sort=False):
    k, n = int(g.ok.sum()), len(g)
    lo, hi = wilson_interval(k, n)
    summary.append(dict(mode=mode, dung=f"{k}/{n}", ci95=f"[{lo:.2f}, {hi:.2f}]", buoc_tb=round(g.n_steps.mean(), 2),
                        loi_tool=int(g.tool_errors.sum()), loi_dinh_dang=int(g.format_errors.sum()),
                        khong_tra_loi=int((g.stopped != "answer").sum()),
                        giay_trung_vi=round(g.seconds.median(), 1), token_prompt_tb=round(g.prompt_tokens.mean())))
print(pd.DataFrame(summary).to_string(index=False))
by_mode = {m: dict(zip(g.id, g.ok)) for m, g in df.groupby("mode", sort=False)}
for a, b in [("react", "native"), ("react", "react_json")]:
    only_a, only_b = discordant(by_mode[a], by_mode[b])
    print(f"{a} đúng mà {b} sai: {only_a} | {b} đúng mà {a} sai: {only_b}")
print(df.pivot_table(index="group", columns="mode", values="ok", aggfunc="sum", sort=False))

In [ ]:
# explore: phân loại lỗi và đọc vài trace
df["failure"] = [classify_failure(r) for r in rows]
print(pd.crosstab(df.failure, df["mode"]))
for mode in MODES:
    print(f"\n===== {mode}: các task sai =====")
    for r in df[(df["mode"] == mode) & ~df.ok].itertuples():
        trace = [(s["tool"], s["args"], (s["observation"] or "")[:60]) for s in r.steps if s["kind"] == "action"]
        print(f"{r.id} [{r.failure}] {r.question}\n   trả lời: {r.answer[:120]!r}\n   tool: {trace}")

In [ ]:
# explore: mở rộng (không chấm): thêm 20 tool gây nhiễu vào registry rồi chạy lại ReAct.
# Câu hỏi: nhiều tool trong context hơn thì agent chọn sai tool nhiều hơn không, và tốn thêm bao nhiêu token?
def make_distractor(name, doc):
    def tool(query: str) -> str:
        return "Dịch vụ tạm thời không khả dụng."
    tool.__name__, tool.__doc__ = name, doc + "\n\nArgs:\n    query: nội dung yêu cầu"
    return tool


DISTRACTORS = [("get_weather", "Thời tiết hiện tại của một thành phố."), ("send_email", "Gửi email."),
               ("create_calendar_event", "Tạo sự kiện trong lịch."), ("translate_text", "Dịch văn bản."),
               ("search_web", "Tìm kiếm trên internet."), ("get_stock_price", "Giá cổ phiếu."),
               ("book_flight", "Đặt vé máy bay."), ("get_news", "Tin tức mới nhất."),
               ("convert_units", "Đổi đơn vị đo lường."), ("get_traffic", "Tình hình giao thông."),
               ("set_reminder", "Đặt nhắc việc."), ("play_music", "Phát nhạc."),
               ("get_movie_showtimes", "Lịch chiếu phim."), ("order_food", "Đặt đồ ăn."),
               ("check_bank_balance", "Số dư tài khoản."), ("get_horoscope", "Tử vi hôm nay."),
               ("find_restaurant", "Tìm nhà hàng."), ("get_exchange_history", "Lịch sử tỷ giá theo ngày."),
               ("track_package", "Theo dõi bưu kiện."), ("get_library_hours", "Giờ mở cửa thư viện.")]
noisy = ToolRegistry()
for fn in bookstore.TOOLS:
    noisy.register(fn)
for name, doc in DISTRACTORS:
    noisy.register(make_distractor(name, doc))
noisy_rows = []
for task_id, group, question, expected in bookstore.TASKS:
    client = OllamaChat()
    res = run_react(client, question, noisy, max_steps=8)
    wrong_tool = sum(1 for s in res.steps if s.kind == "action" and s.tool in dict(DISTRACTORS))
    noisy_rows.append(dict(id=task_id, ok=check_answer(res.answer, expected), wrong_tool=wrong_tool,
                           prompt_tokens=max(client.prompt_tokens)))
noisy_df = pd.DataFrame(noisy_rows)
print("25 tool: đúng", int(noisy_df.ok.sum()), "/ 24 | gọi tool nhiễu:", int(noisy_df.wrong_tool.sum()),
      "| token prompt TB:", round(noisy_df.prompt_tokens.mean()))

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Khoảng tin cậy của ba cách chồng lên nhau tới đâu? Với 24 task, chênh 2 tới 3 task có nói được gì không? Nhìn số cặp
  lệch nhau của `discordant`, không chỉ hiệu hai tỉ lệ.
- Mỗi cách hỏng theo kiểu nào nhiều nhất (`classify_failure`)? Đọc trace: agent tự bịa giá hay đọc sai observation?
  Agent hỏi lại người dùng thay vì gọi tool tiếp là nhãn gì?
- Cách JSON chậm hơn ReAct văn bản bao nhiêu lần, và vì sao (đếm số bước, đọc output)?
- Đổi `seed` (hoặc `temperature` 0.7) rồi chạy lại một cách: tỉ lệ đúng dao động bao nhiêu? So với chênh lệch giữa các
  cách.

Sau đó làm phần trắc nghiệm của bài trong app.